# 2. Data Sources

This section is focused on describing the dataset used in our project, explaining its structure, and loads it for use in the subsequent notebooks

## 2.1 Dataset Overview

**Dataset:** SQL Generator No CoT  
**Link:** Hugging Face - [`AI4DS/sql_generator_no_cot`](https://huggingface.co/datasets/AI4DS/sql_generator_no_cot)  
**Size:** 9,399 examples (training split)  


This dataset is built for training and evaluating text-to-SQL models that are purposely built with **no chain-of-thought (CoT) reasoning**. Each example is a direct prompt + SQL pair, meaning the model must produce the respective SQL output wihtout any reasoning steps. This makes our task a clean supervised sequence-to-sequence problem.

The examples displayed in the dataset are a variet of real-world database scchemas, covering many domains like sports, academia, e-commerce, geography and more.

## 2.2 Column Descriptions

The dataset contains two columns:
| Column | Type | Description |
|---|---|---|
| `prompt` | string | A full system + user prompt given to the model. It is constructed to include a role preamble ("You are a data science expert..."), the database schema (table and column definitions), and the natural language question to answer. |
| `response` | string | The target SQL query output wrapped in a markdown code block (` ```sql ... ``` `). |

The `prompt` field is structured as an instruction-tuning prompt rather than just a raw question.

## 2.3 Loading the Dataset

In [24]:
import pandas as pd

In [25]:
df = pd.read_csv("hf://datasets/AI4DS/sql_generator_no_cot/training_no_cot_dataset.csv")
df

,prompt,response
0,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(DISTINCT T2.teacher_accti...
1,"You are a data science expert.\nBelow, you are...",```sql\nSELECT DISTINCT T2.LongName FROM Count...
2,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(DISTINCT T1.playerID) FRO...
3,"You are a data science expert.\nBelow, you are...",```sql\nSELECT T2.FullName FROM Paper AS T1 IN...
4,"You are a data science expert.\nBelow, you are...",```sql\nSELECT T2.StoreID FROM `Sales Orders` ...
...,...,...
9394,"You are a data science expert.\nBelow, you are...",```sql\nSELECT CASE WHEN CommentIsXml = 0 THEN...
9395,"You are a data science expert.\nBelow, you are...",```sql\nSELECT device_id FROM gender_age WHERE...
9396,"You are a data science expert.\nBelow, you are...",```sql\nSELECT T2.p_id FROM course AS T1 INNER...
9397,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(T1.app_id) FROM app_label...


## 2.4 Dataset Inspection

### 2.4.1 Size and Structure

In [26]:
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print()
df.info()

Shape: (9399, 2)
Columns: ['prompt', 'response']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9399 entries, 0 to 9398
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   prompt    9399 non-null   object
 1   response  9399 non-null   object
dtypes: object(2)
memory usage: 147.0+ KB


### 2.4.2 Missing Value Check

In [27]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
prompt      0
response    0
dtype: int64


### 2.4.3 Example Prompt + Response

In [28]:
idx = 0
print("=== PROMPT ===")
print(df.loc[idx, 'prompt'])
print()
print("=== RESPONSE ===")
print(df.loc[idx, 'response'])

=== PROMPT ===
You are a data science expert.
Below, you are presented with a database schema and a question.
Your task is to read the schema, understand the question, and generate a valid SQLite query to answer the question.

This schema offers an in-depth description of the database's architecture, detailing tables, columns, primary keys, foreign keys, and any pertinent information regarding relationships or constraints. Special attention should be given to the examples listed beside each column, as they directly hint at which columns are relevant to our query.


Database Schema
###
CREATE TABLE donations
(
	donationid TEXT not null primary key,
	projectid TEXT, --
	is_teacher_acct TEXT, -- `is teacher acct` description: whether donor is also a teacher value description: • f: false, it means this donor is not a teacher • t: true, it means this donor is a teacher
	foreign key (projectid) references projects(projectid),
);

CREATE TABLE projects
(
	projectid TEXT not null primary key,


## 2.5 Prompt Structure

Looking at an example raw prompt, we can see that there is a somewhat consistent structure/template that it follows:

```
You are a data science expert.
Below, you are presented with a database schema and a question.
Your task is to read the schema, understand the question, and generate a valid SQLite query to answer it.

Database Schema:
<table definitions with column names and types>

Question: <natural language question>
```
For modelling, there is two parts in tersm fo cleaning the response. The first is extracting the **natural lnanguage question** and the **schema** from the full prompt. Then the next stage is using the `response` column that contains teh SQL answer wrapped in a markdownc ode block, we strip it of its ` ```sql``` ` delimiters during preprocessing.

In [29]:
import re

def extract_question(prompt: str) -> str:
    """Extract the natural language question from the full prompt."""
    match = re.search(r'Question:\s*(.+?)(?:\n|$)', prompt, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else prompt

def extract_sql(response: str) -> str:
    """Strip markdown code block delimiters from the SQL response."""
    match = re.search(r'```sql\s*(.*?)\s*```', response, re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else response.strip()

df['question'] = df['prompt'].apply(extract_question)
df['sql'] = df['response'].apply(extract_sql)

print("Sample extracted question:")
print(df.loc[0, 'question'])
print()
print("Sample extracted SQL:")
print(df.loc[0, 'sql'])

Sample extracted question:
How many teachers have made some type of donation for projects in Chicago?

Sample extracted SQL:
SELECT COUNT(DISTINCT T2.teacher_acctid) FROM donations AS T1 INNER JOIN projects AS T2 ON T1.projectid = T2.projectid WHERE T1.is_teacher_acct = 't' AND T2.school_city = 'Chicago' ;


The processed `df` with `question` and `sql` columns added are exported for use by the following model notebooks.

## 2.6 Data Cleaning

First stage of data cleaning is checking for null values and duplicates across three different cases:
- Exact row duplicates
- Duplicate question + SQL pairs
- Same question mapped to different SQL

Any duplicates that are found are removed before saving

In [30]:
print(f"Rows before cleaning: {len(df):,}")
print()

# Null check
print("Nulls:")
print(df[["prompt", "response", "question", "sql"]].isnull().sum())
print()

# Duplicate check
exact_dups         = df.duplicated().sum()
q_sql_dups         = df.duplicated(subset=["question", "sql"]).sum()
same_q_diff_sql    = df.duplicated(subset=["question"], keep=False).sum() - q_sql_dups
print(f"Exact duplicate rows:          {exact_dups}")
print(f"Duplicate (question, sql) pairs: {q_sql_dups}")
print(f"Same question, different SQL:   {same_q_diff_sql}")
print()

df = df.drop_duplicates(subset=["question", "sql"]).reset_index(drop=True)
print(f"Rows after cleaning: {len(df):,}")

Rows before cleaning: 9,399

Nulls:
prompt      0
response    0
question    0
sql         0
dtype: int64

Exact duplicate rows:          2
Duplicate (question, sql) pairs: 201
Same question, different SQL:   183

Rows after cleaning: 9,198


## 2.7 Saving to Processed Data

In [31]:
import os

# Save processed dataframe for use in subsequent notebooks
os.makedirs("../data", exist_ok=True)
df[["prompt", "question", "response", "sql"]].to_csv("../data/processed.csv", index=False)
print(f"Saved {len(df):,} rows to ../data/processed.csv")

Saved 9,198 rows to ../data/processed.csv
